# RetainSight — Exploratory Data Analysis & Churn Investigation

**Goal:** Explore the customer dataset, understand behavioral patterns, and investigate what drives churn — before building any ML model.

This notebook walks through:
1. Data loading & sanity checks
2. Customer demographics & distributions
3. Revenue & subscription analysis
4. Behavioral event patterns
5. Churn deep-dive — what distinguishes churned customers?
6. Feature correlations & insights for modeling

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

DB_PATH = ROOT / "data" / "retainsight.db"
conn = sqlite3.connect(str(DB_PATH))

customers = pd.read_sql("SELECT * FROM customers", conn)
subscriptions = pd.read_sql("SELECT * FROM subscriptions", conn)
transactions = pd.read_sql("SELECT * FROM transactions", conn)
events = pd.read_sql("SELECT * FROM events", conn)

print(f"Customers:     {len(customers):,}")
print(f"Subscriptions: {len(subscriptions):,}")
print(f"Transactions:  {len(transactions):,}")
print(f"Events:        {len(events):,}")

---
## 1. Data Overview & Sanity Checks

In [ ]:
customers.head()

In [ ]:
customers.info()

In [ ]:
print("Missing values per table:")
print("\nCustomers:")
print(customers.isnull().sum()[customers.isnull().sum() > 0])
print(f"\nSubscriptions:")
print(subscriptions.isnull().sum()[subscriptions.isnull().sum() > 0])
print(f"\nTransactions:")
print(transactions.isnull().sum().sum(), "total nulls")
print(f"\nEvents:")
print(events.isnull().sum()[events.isnull().sum() > 0])

In [ ]:
churn_rate = customers["is_churned"].mean()
print(f"Overall churn rate: {churn_rate:.1%}")
print(f"Churned: {customers['is_churned'].sum():,} / {len(customers):,}")

---
## 2. Customer Demographics

In [ ]:
fig = make_subplots(rows=2, cols=2,
    subplot_titles=("Age Distribution", "Gender Split",
                    "Customers by Country", "Acquisition Channel"))

fig.add_trace(go.Histogram(x=customers["age"], nbinsx=30,
    marker_color="#4F46E5", name="Age"), row=1, col=1)

gender_counts = customers["gender"].value_counts()
fig.add_trace(go.Bar(x=gender_counts.index, y=gender_counts.values,
    marker_color=["#4F46E5", "#A78BFA", "#C4B5FD"], name="Gender"), row=1, col=2)

country_counts = customers["country"].value_counts()
fig.add_trace(go.Bar(x=country_counts.index, y=country_counts.values,
    marker_color="#7C3AED", name="Country"), row=2, col=1)

channel_counts = customers["acquisition_channel"].value_counts()
fig.add_trace(go.Bar(x=channel_counts.index, y=channel_counts.values,
    marker_color="#10B981", name="Channel"), row=2, col=2)

fig.update_layout(height=600, showlegend=False,
    title_text="Customer Demographics Overview")
fig.show()

In [ ]:
churn_by_country = customers.groupby("country")["is_churned"].agg(["mean", "count"])
churn_by_country.columns = ["churn_rate", "total"]
churn_by_country = churn_by_country.sort_values("churn_rate", ascending=False)

fig = px.bar(churn_by_country.reset_index(), x="country", y="churn_rate",
    text=churn_by_country["total"].values,
    labels={"churn_rate": "Churn Rate", "country": "Country"},
    title="Churn Rate by Country (numbers = customer count)",
    color="churn_rate", color_continuous_scale="RdYlGn_r")
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_tickformat=".0%")
fig.show()

---
## 3. Revenue & Subscription Analysis

In [ ]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])
transactions["month"] = transactions["transaction_date"].dt.to_period("M").astype(str)

monthly_rev = transactions.groupby("month")["amount"].agg(["sum", "count", "mean"]).reset_index()
monthly_rev.columns = ["month", "revenue", "transactions", "avg_amount"]

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=monthly_rev["month"], y=monthly_rev["revenue"],
    name="Revenue", marker_color="#4F46E5"), secondary_y=False)
fig.add_trace(go.Scatter(x=monthly_rev["month"], y=monthly_rev["transactions"],
    name="# Transactions", line=dict(color="#EF4444", width=2)), secondary_y=True)
fig.update_layout(title="Monthly Revenue & Transaction Volume", height=400)
fig.update_yaxes(title_text="Revenue ($)", secondary_y=False)
fig.update_yaxes(title_text="Transaction Count", secondary_y=True)
fig.show()

In [ ]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Revenue by Product Category", "Transaction Amount Distribution"))

cat_rev = transactions.groupby("product_category")["amount"].sum().sort_values(ascending=False)
fig.add_trace(go.Bar(x=cat_rev.index, y=cat_rev.values,
    marker_color=["#4F46E5", "#7C3AED", "#A78BFA", "#C4B5FD"]), row=1, col=1)

fig.add_trace(go.Histogram(x=transactions["amount"], nbinsx=50,
    marker_color="#10B981"), row=1, col=2)

fig.update_layout(height=400, showlegend=False,
    title_text="Transaction Analysis")
fig.show()

In [ ]:
plan_dist = subscriptions.groupby("plan").agg(
    count=("subscription_id", "count"),
    avg_mrr=("mrr", "mean"),
    total_mrr=("mrr", "sum")
).reindex(["free", "starter", "pro", "enterprise"])

print("Subscription plan breakdown:")
plan_dist

In [ ]:
status_counts = subscriptions["status"].value_counts()

fig = px.pie(values=status_counts.values, names=status_counts.index,
    title="Subscription Status Distribution",
    color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_traces(textposition="inside", textinfo="percent+label")
fig.show()

---
## 4. Behavioral Event Patterns

In [ ]:
event_type_counts = events["event_type"].value_counts()
print("Event type distribution:")
print(event_type_counts)
print(f"\nTotal events: {len(events):,}")
print(f"Unique customers with events: {events['customer_id'].nunique():,}")

In [ ]:
events["event_date"] = pd.to_datetime(events["event_date"])

event_agg = events.merge(
    customers[["customer_id", "is_churned"]], on="customer_id"
)
event_agg["status"] = event_agg["is_churned"].map({0: "Active", 1: "Churned"})

per_customer_events = event_agg.groupby(["customer_id", "status"]).agg(
    total_events=("event_id", "count"),
    logins=("event_type", lambda x: (x == "login").sum()),
    feature_uses=("event_type", lambda x: (x == "feature_use").sum()),
    support_tickets=("event_type", lambda x: (x == "support_ticket").sum()),
    page_views=("event_type", lambda x: (x == "page_view").sum()),
).reset_index()

print("Avg events per customer by status:")
per_customer_events.groupby("status")[["total_events", "logins", "feature_uses",
    "support_tickets", "page_views"]].mean().round(1)

In [ ]:
melted = per_customer_events.melt(
    id_vars=["customer_id", "status"],
    value_vars=["logins", "feature_uses", "support_tickets", "page_views"],
    var_name="event_type", value_name="count"
)

fig = px.box(melted, x="event_type", y="count", color="status",
    color_discrete_map={"Active": "#4F46E5", "Churned": "#EF4444"},
    title="Event Distribution: Active vs Churned Customers",
    labels={"event_type": "Event Type", "count": "Count per Customer"})
fig.update_layout(height=500)
fig.show()

---
## 5. Churn Deep Dive

The key question: **what makes churned customers different from active ones?**

In [ ]:
customers["signup_date"] = pd.to_datetime(customers["signup_date"])
customers["churn_date"] = pd.to_datetime(customers["churn_date"])

spend_per_customer = transactions.groupby("customer_id")["amount"].sum().reset_index()
spend_per_customer.columns = ["customer_id", "total_spend"]

txn_count = transactions.groupby("customer_id")["transaction_id"].count().reset_index()
txn_count.columns = ["customer_id", "num_transactions"]

cust_full = customers.merge(spend_per_customer, on="customer_id", how="left")
cust_full = cust_full.merge(txn_count, on="customer_id", how="left")
cust_full = cust_full.merge(
    per_customer_events[["customer_id", "total_events", "logins", "feature_uses"]],
    on="customer_id", how="left"
)
cust_full = cust_full.fillna(0)
cust_full["status"] = cust_full["is_churned"].map({0: "Active", 1: "Churned"})

ref_date = pd.Timestamp("2025-03-31")
cust_full["tenure_days"] = (ref_date - cust_full["signup_date"]).dt.days

In [ ]:
print("Churned vs Active — Key Metrics Comparison")
print("=" * 50)
comparison = cust_full.groupby("status").agg(
    count=("customer_id", "count"),
    avg_spend=("total_spend", "mean"),
    median_spend=("total_spend", "median"),
    avg_transactions=("num_transactions", "mean"),
    avg_events=("total_events", "mean"),
    avg_logins=("logins", "mean"),
    avg_feature_uses=("feature_uses", "mean"),
    avg_tenure=("tenure_days", "mean"),
    avg_age=("age", "mean"),
).round(1).T
comparison

In [ ]:
fig = make_subplots(rows=2, cols=2,
    subplot_titles=("Total Spend", "Total Events",
                    "Number of Transactions", "Tenure (days)"))

for i, col in enumerate(["total_spend", "total_events", "num_transactions", "tenure_days"]):
    row, c = divmod(i, 2)
    for status, color in [("Active", "#4F46E5"), ("Churned", "#EF4444")]:
        subset = cust_full[cust_full["status"] == status]
        fig.add_trace(go.Histogram(
            x=subset[col], name=status, marker_color=color,
            opacity=0.7, nbinsx=40, showlegend=(i == 0)
        ), row=row+1, col=c+1)

fig.update_layout(barmode="overlay", height=600,
    title_text="Distribution Comparison: Active vs Churned")
fig.show()

In [ ]:
churn_by_channel = customers.groupby("acquisition_channel")["is_churned"].agg(["mean", "count"]).reset_index()
churn_by_channel.columns = ["channel", "churn_rate", "total"]

churn_by_gender = customers.groupby("gender")["is_churned"].agg(["mean", "count"]).reset_index()
churn_by_gender.columns = ["gender", "churn_rate", "total"]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Churn Rate by Acquisition Channel", "Churn Rate by Gender"))

fig.add_trace(go.Bar(
    x=churn_by_channel["channel"], y=churn_by_channel["churn_rate"],
    marker_color="#7C3AED", text=churn_by_channel["churn_rate"].apply(lambda x: f"{x:.1%}"),
    textposition="outside"
), row=1, col=1)

fig.add_trace(go.Bar(
    x=churn_by_gender["gender"], y=churn_by_gender["churn_rate"],
    marker_color="#10B981", text=churn_by_gender["churn_rate"].apply(lambda x: f"{x:.1%}"),
    textposition="outside"
), row=1, col=2)

fig.update_layout(height=400, showlegend=False, title_text="Churn Rate Breakdown")
fig.update_yaxes(tickformat=".0%")
fig.show()

In [ ]:
fig = px.scatter(cust_full, x="total_spend", y="total_events",
    color="status",
    color_discrete_map={"Active": "#4F46E5", "Churned": "#EF4444"},
    opacity=0.5,
    title="Spend vs Engagement — Churned customers cluster in the low-low corner",
    labels={"total_spend": "Total Spend ($)", "total_events": "Total Events"})
fig.update_layout(height=500)
fig.show()

**Key observation:** Churned customers clearly cluster in the bottom-left — low spend AND low engagement. This suggests both behavioral and transactional features will be strong predictors for the churn model.

---
## 6. Feature Correlations

In [ ]:
corr_cols = ["is_churned", "total_spend", "num_transactions", "total_events",
             "logins", "feature_uses", "tenure_days", "age"]
corr_matrix = cust_full[corr_cols].corr()

fig = px.imshow(corr_matrix,
    text_auto=".2f",
    color_continuous_scale="RdBu_r",
    title="Feature Correlation Matrix",
    aspect="auto")
fig.update_layout(height=550)
fig.show()

In [ ]:
churn_corr = corr_matrix["is_churned"].drop("is_churned").sort_values()

fig = px.bar(x=churn_corr.values, y=churn_corr.index, orientation="h",
    title="Correlation with Churn (negative = reduces churn risk)",
    labels={"x": "Correlation", "y": "Feature"},
    color=churn_corr.values,
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=0)
fig.update_layout(height=400)
fig.show()

---
## 7. Cohort Retention Analysis

In [ ]:
from src.analytics.queries import cohort_retention

cohort_df = cohort_retention()
pivot = cohort_df.pivot_table(index="cohort", columns="period",
    values="retention_rate")
pivot = pivot[[c for c in sorted(pivot.columns) if c <= 12]]

fig = px.imshow(pivot,
    labels=dict(x="Months Since Signup", y="Signup Cohort", color="Retention %"),
    color_continuous_scale="Blues",
    aspect="auto", text_auto=".0f",
    title="Cohort Retention Heatmap")
fig.update_layout(height=600)
fig.show()

---
## 8. Conversion Funnel

In [ ]:
from src.analytics.queries import conversion_funnel

funnel = conversion_funnel()

fig = go.Figure(go.Funnel(
    y=funnel["stage"],
    x=funnel["users"],
    textinfo="value+percent initial",
    marker=dict(color=["#4F46E5", "#7C3AED", "#A78BFA", "#C4B5FD", "#DDD6FE"])))
fig.update_layout(title="Customer Conversion Funnel", height=400)
fig.show()

---
## Key Takeaways

1. **Churn rate is ~25%** — significant enough to warrant a retention strategy
2. **Engagement is the strongest churn signal** — churned customers have dramatically fewer events, logins, and feature uses
3. **Spend and engagement are correlated** — but engagement drops *before* spend does, making it a leading indicator
4. **Acquisition channel and demographics show minimal churn variation** — churn is primarily behavioral, not demographic
5. **The feature matrix should prioritize recency and engagement signals** — events_last_30d, days_since_last_event, and feature_use_count will likely be top predictors

These insights informed the feature engineering in `src/ml/feature_engineering.py` and the intervention thresholds in `src/decision_engine/engine.py`.

In [ ]:
conn.close()
print("Analysis complete.")